# 🧠 CDM RAG Assistant
**Retrieval-Augmented Generation for Clinical Data Management Documents**

This notebook lets you ask questions about provided CDM/CDS documents and get answers with source citations.

### 📋 How to use this notebook
Run each block **from top to bottom**, in order. You only need to re-run from Block 4 onwards after adding new documents.

| Block | What it does | Run again? |
|-------|-------------|------------|
| 1 | Install packages | Only once |
| 2 | Import libraries | Each session |
| 3 | Load documents (PDF, XLSX, JSON, XML, CSV, DOCX) | When you add files |
| 4 | Chunk text | When you add files |
| 5 | Build search index | When you add files |
| 6 | Connect to Ollama LLM | Each session |
| 7 | Ask a question | Anytime! |
| 8 | Interactive Q&A loop | Anytime! |

## 📦 Block 1 — Install Required Packages
Run this once. It installs everything the notebook needs.

> ⚠️ **Before running Block 6**, you also need to install Ollama separately:
> 1. Go to [https://ollama.com](https://ollama.com) and download the Mac app
> 2. Open Terminal and run: `ollama pull mistral`
> 3. Ollama will then run in the background automatically

In [1]:
# Install all required Python packages
# M4 Mac fix: use chromadb instead of faiss-cpu (no ARM64 conflict)
# This may take a few minutes the first time
import sys

!{sys.executable} -m pip install --upgrade pip --quiet
!{sys.executable} -m pip install sentence-transformers chromadb pypdf openpyxl python-docx requests tqdm --quiet
!{sys.executable} -m pip install --upgrade jupyter ipywidgets --quiet

print("✅ All packages installed!")

✅ All packages installed!


## 📚 Block 2 — Import Libraries
This loads all the tools we need into memory. Run this at the start of every session.

In [2]:
import os
import json
import csv
import xml.etree.ElementTree as ET
import requests
import numpy as np
import chromadb                          # replaces faiss — works on M4

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm
import openpyxl
import docx  # python-docx

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


## 📁 Block 3 — Load Documents

**👇 Only change this line:** set `DOCS_FOLDER` to the path of your documents folder.

Supported file types:
- 📄 **PDF** — extracts text page by page
- 📊 **XLSX** — reads each sheet row by row
- 📝 **DOCX** — reads paragraphs
- 🗂️ **JSON** — reads structured data
- 🔖 **XML** — reads element text
- 📋 **CSV** — reads rows as text
- 📃 **TXT** — reads plain text

In [3]:
# ============================================================
# 🔴 CHANGE THIS to your documents folder path
# ============================================================
DOCS_FOLDER = "../data/raw/"
# ============================================================

# --- Helper: detect heading from PDF page text ---
def extract_heading(text):
    """Try to find a heading on the page (short ALL-CAPS line)."""
    for line in text.split("\n"):
        stripped = line.strip()
        if 4 < len(stripped) < 80 and stripped.isupper():
            return stripped
    return "Unknown"

# --- Loader functions per file type ---
def load_pdf(filepath):
    """Load a PDF file. Returns one entry per page."""
    docs = []
    filename = os.path.basename(filepath)
    try:
        reader = PdfReader(filepath)
        for i, page in enumerate(reader.pages):
            text = page.extract_text()
            if text and text.strip():
                docs.append({
                    "text": text.strip(),
                    "source": filename,
                    "page": i + 1,
                    "heading": extract_heading(text)
                })
    except Exception as e:
        print(f"  ❌ Error reading PDF {filename}: {e}")
    return docs

def load_xlsx(filepath):
    """Load an Excel file. Returns one entry per sheet (rows joined as text)."""
    docs = []
    filename = os.path.basename(filepath)
    try:
        wb = openpyxl.load_workbook(filepath, read_only=True, data_only=True)
        for sheet_name in wb.sheetnames:
            ws = wb[sheet_name]
            rows_text = []
            for row in ws.iter_rows(values_only=True):
                row_str = " | ".join(str(cell) for cell in row if cell is not None)
                if row_str.strip():
                    rows_text.append(row_str)
            if rows_text:
                docs.append({
                    "text": "\n".join(rows_text),
                    "source": filename,
                    "page": sheet_name,   # use sheet name as 'page'
                    "heading": f"Sheet: {sheet_name}"
                })
    except Exception as e:
        print(f"  ❌ Error reading XLSX {filename}: {e}")
    return docs

def load_docx(filepath):
    """Load a Word document. Returns one entry per document."""
    docs = []
    filename = os.path.basename(filepath)
    try:
        doc = docx.Document(filepath)
        paragraphs = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
        full_text = "\n".join(paragraphs)
        if full_text:
            docs.append({
                "text": full_text,
                "source": filename,
                "page": 1,
                "heading": paragraphs[0][:80] if paragraphs else "Unknown"
            })
    except Exception as e:
        print(f"  ❌ Error reading DOCX {filename}: {e}")
    return docs

def load_json(filepath):
    """Load a JSON file. Converts to text."""
    docs = []
    filename = os.path.basename(filepath)
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)
        text = json.dumps(data, indent=2, ensure_ascii=False)
        docs.append({
            "text": text,
            "source": filename,
            "page": 1,
            "heading": "JSON Document"
        })
    except Exception as e:
        print(f"  ❌ Error reading JSON {filename}: {e}")
    return docs

def load_xml(filepath):
    """Load an XML file. Extracts all text content."""
    docs = []
    filename = os.path.basename(filepath)
    try:
        tree = ET.parse(filepath)
        root = tree.getroot()
        # Collect all text from all elements
        texts = []
        for elem in root.iter():
            if elem.text and elem.text.strip():
                texts.append(f"{elem.tag}: {elem.text.strip()}")
        full_text = "\n".join(texts)
        if full_text:
            docs.append({
                "text": full_text,
                "source": filename,
                "page": 1,
                "heading": f"XML Root: {root.tag}"
            })
    except Exception as e:
        print(f"  ❌ Error reading XML {filename}: {e}")
    return docs

def load_csv(filepath):
    """Load a CSV file. Returns one entry for the whole file."""
    docs = []
    filename = os.path.basename(filepath)
    try:
        with open(filepath, "r", encoding="utf-8", errors="replace") as f:
            reader = csv.reader(f)
            rows = [" | ".join(row) for row in reader if any(cell.strip() for cell in row)]
        full_text = "\n".join(rows)
        if full_text:
            docs.append({
                "text": full_text,
                "source": filename,
                "page": 1,
                "heading": "CSV Document"
            })
    except Exception as e:
        print(f"  ❌ Error reading CSV {filename}: {e}")
    return docs

def load_txt(filepath):
    """Load a plain text file."""
    docs = []
    filename = os.path.basename(filepath)
    try:
        with open(filepath, "r", encoding="utf-8", errors="replace") as f:
            text = f.read().strip()
        if text:
            docs.append({
                "text": text,
                "source": filename,
                "page": 1,
                "heading": "Text Document"
            })
    except Exception as e:
        print(f"  ❌ Error reading TXT {filename}: {e}")
    return docs

# --- Main loader: scans folder and picks the right loader ---
LOADERS = {
    ".pdf":  load_pdf,
    ".xlsx": load_xlsx,
    ".xls":  load_xlsx,
    ".docx": load_docx,
    ".json": load_json,
    ".xml":  load_xml,
    ".csv":  load_csv,
    ".txt":  load_txt,
}

def load_all_documents(folder_path):
    """Walk a folder and load all supported file types."""
    all_docs = []
    skipped = []

    if not os.path.exists(folder_path):
        print(f"❌ Folder not found: {folder_path}")
        print("   Please update DOCS_FOLDER above to your actual documents folder.")
        return all_docs

    # Walk folder recursively (includes subfolders)
    for root, dirs, files in os.walk(folder_path):
        for filename in sorted(files):
            ext = os.path.splitext(filename)[1].lower()
            if ext in LOADERS:
                full_path = os.path.join(root, filename)
                print(f"  📄 Loading: {filename}")
                docs = LOADERS[ext](full_path)
                print(f"     → {len(docs)} section(s) found")
                all_docs.extend(docs)
            elif not filename.startswith("."):  # Ignore hidden files
                skipped.append(filename)

    print(f"\n✅ Total sections loaded: {len(all_docs)}")
    if skipped:
        print(f"⚠️  Skipped (unsupported format): {', '.join(skipped)}")

    return all_docs

# Run the loader
print(f"🔍 Scanning folder: {DOCS_FOLDER}\n")
docs = load_all_documents(DOCS_FOLDER)

🔍 Scanning folder: ../data/raw/

  📄 Loading: 2019_Evolution-of-CDM-to-CDS-Part-1-Drivers.pdf
     → 23 section(s) found
  📄 Loading: 2020_Evolution-of-CDM-to-CDS-Part-2-Technology-Enablers.pdf
     → 31 section(s) found
  📄 Loading: 2020_Evolution-of-CDM-to-CDS-Part-3-Evolution-of-CDM-Role.pdf
     → 31 section(s) found
  📄 Loading: SCDM-Position-Paper-Evolution-into-Clinical-to-Data-Science-V9.0.pdf
     → 21 section(s) found

✅ Total sections loaded: 106


## ✂️ Block 4 — Split Text into Chunks

Long documents are split into smaller overlapping pieces so the search engine can find precise answers.

- `CHUNK_SIZE` = how many words per chunk (500 is a good default)
- `CHUNK_OVERLAP` = how many words overlap between chunks (helps avoid cutting answers in half)

In [4]:
# ============================================================
# Settings — you can leave these as-is to start
# ============================================================
CHUNK_SIZE    = 500   # words per chunk
CHUNK_OVERLAP = 50    # words of overlap between chunks
# ============================================================

chunks   = []  # stores the text of each chunk
metadata = []  # stores source, page, heading for each chunk

for doc in docs:
    words = doc["text"].split()
    step  = CHUNK_SIZE - CHUNK_OVERLAP  # how far to step forward each time

    for i in range(0, len(words), step):
        chunk_words = words[i : i + CHUNK_SIZE]
        chunk_text  = " ".join(chunk_words)
        if chunk_text.strip():
            chunks.append(chunk_text)
            metadata.append({
                "source":  doc["source"],
                "page":    doc["page"],
                "heading": doc["heading"]
            })

print(f"✅ Created {len(chunks)} chunks from {len(docs)} document sections")
print(f"   (chunk size: {CHUNK_SIZE} words, overlap: {CHUNK_OVERLAP} words)")

✅ Created 160 chunks from 106 document sections
   (chunk size: 500 words, overlap: 50 words)


## 🧠 Block 5 — Build the Search Index

This converts all your text chunks into numbers (called *embeddings*) that the computer can compare mathematically. Then it stores them in a fast search index called FAISS.

**This may take several minutes** the first time — it's doing a lot of maths!

The model `all-MiniLM-L6-v2` is small, fast, and works well for document search.

In [5]:
# ── M4 Mac fix ──────────────────────────────────────────────────────────────
# 1. device='mps'  → uses Apple Silicon GPU (fast). Falls back to 'cpu'.
# 2. batch_size=32  → was 2 (too small, causes memory thrashing on M4).
# 3. chromadb       → replaces faiss-cpu (which crashes on ARM64).
# ─────────────────────────────────────────────────────────────────────────────

# Step 1: Load the embedding model
print("⏳ Loading embedding model (downloads once, then cached)...")

import torch
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"   Using device: {device}")

embed_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
print("✅ Embedding model ready!")

# Step 2: Generate embeddings for all chunks
print(f"\n⏳ Encoding {len(chunks)} chunks into embeddings...")
embeddings = embed_model.encode(
    chunks,
    show_progress_bar=True,
    batch_size=32,          # M4 fix: was 2 — larger batches are stable on MPS
    convert_to_numpy=True
)
embeddings = embeddings.astype("float32")
print(f"✅ Embeddings created: {embeddings.shape} (chunks × dimensions)")

# Step 3: Build ChromaDB index (replaces FAISS)
print("\n⏳ Building search index with ChromaDB...")
chroma_client     = chromadb.Client()

# Delete old collection if re-running this block
try:
    chroma_client.delete_collection("rag_docs")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="rag_docs",
    metadata={"hnsw:space": "cosine"}   # cosine similarity (same quality as L2 here)
)

# Add all chunks + their embeddings + metadata
collection.add(
    ids        = [str(i) for i in range(len(chunks))],
    embeddings = embeddings.tolist(),
    documents  = chunks,
    metadatas  = metadata
)

print(f"✅ Search index ready with {collection.count()} vectors")

⏳ Loading embedding model (downloads once, then cached)...
   Using device: mps


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model ready!

⏳ Encoding 160 chunks into embeddings...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

✅ Embeddings created: (160, 384) (chunks × dimensions)

⏳ Building search index with ChromaDB...
✅ Search index ready with 160 vectors


## 🤖 Block 6 — Connect to Ollama (Local AI)

This connects to Ollama running on your Mac. Ollama must be installed and running before this works.

**Quick setup (only needed once):**
1. Download Ollama from [https://ollama.com](https://ollama.com)
2. Open Terminal and run: `ollama pull mistral`
3. Ollama runs automatically in the background

**Which model to use?**
- `mistral` — best balance of speed and quality (recommended ✅)
- `llama3.2` — also great, slightly slower
- `phi3` — fastest, good for quick tests

In [6]:
# ============================================================
# Settings
# ============================================================
OLLAMA_MODEL = "mistral"         # Change to llama3.2 or phi3 if preferred
OLLAMA_URL   = "http://localhost:11434/api/generate"
# ============================================================


def ask_ollama(prompt, model=OLLAMA_MODEL, temperature=0.3):
    """
    Send a prompt to Ollama and return the response text.

    Args:
        prompt (str): The full prompt to send
        model (str): Which Ollama model to use
        temperature (float): How creative/random the answer is (0 = factual, 1 = creative)
    Returns:
        str: The LLM's answer
    """
    try:
        response = requests.post(
            OLLAMA_URL,
            json={
                "model":  model,
                "prompt": prompt,
                "stream": False,
                "options": {"temperature": temperature}
            },
            timeout=120  # wait up to 2 minutes for a response
        )
        response.raise_for_status()
        return response.json().get("response", "").strip()

    except requests.exceptions.ConnectionError:
        return (
            "❌ Cannot connect to Ollama.\n"
            "   Make sure Ollama is installed and running.\n"
            "   Download from: https://ollama.com\n"
            f"  Then in Terminal run: ollama pull {OLLAMA_MODEL}"
        )
    except Exception as e:
        return f"❌ Error: {e}"


# Test the connection
print(f"🔗 Testing connection to Ollama (model: {OLLAMA_MODEL})...")
test_reply = ask_ollama("Reply with exactly: 'Ollama is working!'")
print(f"Response: {test_reply}")

if "working" in test_reply.lower() or "ollama" in test_reply.lower():
    print("\n✅ Ollama is connected and ready!")
else:
    print("\n⚠️  Got a response — Ollama seems to be running.")

🔗 Testing connection to Ollama (model: mistral)...
Response: Ollama is working!

✅ Ollama is connected and ready!


## 🔍 Block 7 — Search + Ask a Question

This is the core of the RAG system:
1. Your question is converted to an embedding
2. FAISS finds the most relevant chunks from your documents
3. Those chunks are sent to Ollama as context
4. Ollama answers your question and cites the sources

**👇 Change `MY_QUESTION` to whatever you want to ask!**

In [7]:
# ============================================================
# 👇 Type your question here
# ============================================================
MY_QUESTION = "What skills does a Clinical Data Manager need to transition to Data Science?"
TOP_K       = 5   # How many document chunks to retrieve (3–7 is usually best)
# ============================================================


def search_documents(question, top_k=5):
    """
    Search the document index for the most relevant chunks.
    Updated to use ChromaDB instead of FAISS.
    """
    query_embedding = embed_model.encode([question]).astype("float32")

    results = collection.query(
        query_embeddings = query_embedding.tolist(),
        n_results         = top_k
    )

    output = []
    for i in range(len(results["ids"][0])):
        distance = results["distances"][0][i]   # cosine distance (lower = better)
        meta     = results["metadatas"][0][i]
        text     = results["documents"][0][i]
        output.append({
            "text":     text,
            "source":   meta.get("source", "unknown"),
            "page":     meta.get("page", "?"),
            "heading":  meta.get("heading", "?"),
            "distance": float(distance)
        })
    return output


def build_prompt(question, retrieved_chunks):
    """
    Build the prompt that gets sent to the LLM.
    Includes the question and the retrieved document context.
    """
    context_parts = []
    for i, chunk in enumerate(retrieved_chunks, 1):
        context_parts.append(
            f"[Source {i}] Document: {chunk['source']} | "
            f"Page/Section: {chunk['page']} | "
            f"Heading: {chunk['heading']}\n"
            f"{chunk['text']}"
        )
    context = "\n\n".join(context_parts)

    prompt = f"""You are an expert assistant in Clinical Data Management (CDM) and Clinical Data Science (CDS).

Answer the question below using ONLY the document excerpts provided as context.
- Be concise and factual
- Always cite your sources using the format: (Source: <filename>, Page: <page>)
- If the context does not contain enough information to answer, say so clearly
- Do not make up information

=== DOCUMENT CONTEXT ===
{context}

=== QUESTION ===
{question}

=== ANSWER ==="""
    return prompt


def ask_question(question, top_k=5, show_sources=True):
    """
    Full RAG pipeline: search documents → build prompt → get LLM answer.
    """
    print(f"\n🔍 Searching documents for: '{question}'")

    # Step 1: Retrieve relevant chunks
    retrieved = search_documents(question, top_k=top_k)
    print(f"   Found {len(retrieved)} relevant sections")

    # Step 2: Build prompt and ask the LLM
    print(f"   Asking {OLLAMA_MODEL}...")
    prompt = build_prompt(question, retrieved)
    answer = ask_ollama(prompt)

    # Step 3: Display results
    print("\n" + "="*60)
    print("💬 ANSWER")
    print("="*60)
    print(answer)

    if show_sources:
        print("\n" + "-"*60)
        print("📚 SOURCES USED")
        print("-"*60)
        for i, r in enumerate(retrieved, 1):
            relevance = "🟢" if r["distance"] < 0.3 else "🟡" if r["distance"] < 0.6 else "🔴"
            print(f"  {relevance} [{i}] {r['source']} | Page: {r['page']} | Heading: {r['heading']}")
            print(f"       Relevance score: {r['distance']:.4f} (lower = more relevant)")
            print(f"       Preview: {r['text'][:200]}...")
            print()

    return answer


# Run the question!
answer = ask_question(MY_QUESTION, top_k=TOP_K)


🔍 Searching documents for: 'What skills does a Clinical Data Manager need to transition to Data Science?'
   Found 5 relevant sections
   Asking mistral...

💬 ANSWER
A Clinical Data Manager needs to build on the core CDM skillsets and focus on emerging opportunities offered by technology, regulations, and Clinical Development strategies. Some of the emerging skillsets for transitioning to Clinical Data Science include:

1. Robust critical thinking and process knowledge (Source 2, Page 20)
2. Broader cross-functional collaboration (Source 2, Page 20)
3. Ability to align the flows of data with the needs of next-generation clinical protocols (Source 2, Page 20)
4. Deep knowledge of data including understanding the implications of data context, quality, source, amount, and workflow (Source 2, Page 20)
5. Advanced analytical and technical skills to interrogate and mine high volumes of data from a variety of sources (Source 2, Page 20)
6. High-level understanding of Artificial Intelligence 

## 💬 Block 8 — Interactive Q&A Loop

This lets you ask multiple questions without re-running cells.
Type your question and press Enter. Type `quit` or `exit` to stop.

In [ ]:
print("💬 Interactive Q&A — type your question and press Enter")
print("   Type 'quit' or 'exit' to stop\n")

while True:
    question = input("\n❓ Your question: ").strip()

    if not question:
        print("   ⚠️  Please type a question.")
        continue

    if question.lower() in ("quit", "exit", "stop", "q"):
        print("\n👋 Goodbye! Come back when you have more questions.")
        break

    ask_question(question, top_k=5)

💬 Interactive Q&A — type your question and press Enter
   Type 'quit' or 'exit' to stop


🔍 Searching documents for: 'I have extensive CDM background of over 25 years and am now learning about AI and data science - what should I focus on learning?'
   Found 5 relevant sections
   Asking mistral...

💬 ANSWER
Based on the provided document context, to focus your learning as an experienced Clinical Data Management (CDM) professional transitioning into Clinical Data Science (CDS), you should concentrate on the following areas:

1. Advanced analytics and data interrogation methods (Source 4, Page/Section: 12)
2. Machine Learning (ML) methodologies (Sources 1, 3, and 4)
3. New research methodology such as adaptive and master protocols (Sources 1 and 4)
4. Decentralized clinical trials approaches and technologies (Sources 1 and 4)
5. Risk-based methodologies and regulations (Source 1)
6. Understanding of new data concepts like sequenced data, unstructured data, and data mining (Source 1)
7. H

---
## 📖 Quick Reference

### Understanding relevance scores (ChromaDB cosine distance)
The distance scores tell you how closely a document matched your question:
- 🟢 **Under 0.3** — very relevant
- 🟡 **0.3 – 0.6** — somewhat relevant
- 🔴 **Over 0.6** — less relevant (may not be useful)

### Tips for better answers
- Ask specific questions: *"What does Part 3 say about the CDM role evolution?"* works better than *"tell me about CDM"*
- If the answer seems wrong, try rephrasing your question
- Increase `TOP_K` to 7 or 8 if answers seem incomplete
- Decrease `CHUNK_SIZE` to 300 if you want more precise source citations
